# Concurrent writes to native Iceberg tables

Native Apache Iceberg tables in AIDP Workbench use optimistic concurrency: each writer prepares a new snapshot and commits it only if nobody else changed the data it depends on. This notebook shows what that means when several writers update the same table at the same time:

- Concurrent **appends** commit safely.
- Concurrent **MERGE / UPDATE / DELETE** on overlapping rows conflict. Iceberg rejects the losing commits so the table stays correct, but your code must retry them.
- A small retry helper makes overlapping writers succeed.

In tests on AIDP Workbench with 3 parallel writers, the results were:

| Writers | Appends | Overlapping MERGE, no retry | Overlapping MERGE, with retry |
|---|---|---|---|
| 3 tasks, one cluster | 45/45 committed | 7/15 committed | 15/15 committed (up to 5 retries) |
| 3 tasks, two clusters | 45/45 committed | 5/15 committed | 14/15 committed with 5 retries; use more |

The table data was correct in every run: failed commits left nothing behind.

New to native Iceberg tables? Start with [Native Apache Iceberg Table](../../../getting-started/Native_Iceberg_Table.ipynb).

**Prerequisites:** an AIDP Workbench notebook attached to a running Spark cluster, and permission to create a schema in a Standard catalog.

## 1. Create a demo table

The schema name is unique per run, so the notebook does not touch earlier runs.

In [ ]:
import uuid

CATALOG_NAME = "default"
SCHEMA_NAME = f"iceberg_concurrency_{uuid.uuid4().hex[:8]}"
TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.orders"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE TABLE {TABLE} (order_id BIGINT, status STRING, updated_by STRING) USING ICEBERG")
spark.sql(f"INSERT INTO {TABLE} SELECT id, 'NEW', 'seed' FROM range(20)")
print("Created", TABLE)

## 2. A retry helper for commit conflicts

Retry only commit conflicts, and fail immediately on any other error. A random wait (jitter) between attempts keeps writers from colliding again in lockstep.

Across clusters, 5 attempts were not always enough for 3 writers, so the default here is 10.

In [ ]:
import random
import time

_CONFLICT_MARKERS = ("CommitFailedException", "ValidationException", "conflicting files", "already exists")


def run_with_retry(statement, max_attempts=10, base_wait_seconds=0.5):
    """Run a SQL statement, retrying Iceberg commit conflicts. Returns the number of retries used."""
    for attempt in range(max_attempts):
        try:
            spark.sql(statement)
            return attempt
        except Exception as error:
            message = str(error)
            if not any(marker in message for marker in _CONFLICT_MARKERS) or attempt == max_attempts - 1:
                raise
            time.sleep(random.uniform(base_wait_seconds, base_wait_seconds * 4) * (attempt + 1))

## 3. Concurrent appends

Three writers each append 5 new rows at the same time. Appends don't depend on each other's data, so every commit succeeds.

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def append_writer(writer_id):
    for i in range(5):
        spark.sql(f"INSERT INTO {TABLE} VALUES ({1000 * writer_id + i}, 'NEW', 'writer_{writer_id}')")
    return writer_id


with ThreadPoolExecutor(max_workers=3) as pool:
    list(pool.map(append_writer, [1, 2, 3]))

rows, distinct = spark.sql(f"SELECT count(*), count(DISTINCT order_id) FROM {TABLE}").first()
print(f"rows={rows} (expected 35), duplicates={rows - distinct}")

## 4. Concurrent MERGE on the same rows, without retries

Three writers each run 3 MERGE statements that update the same 20 seed rows. When two writers change the same rows, Iceberg accepts the first commit and rejects the other. Expect some failures here.

In [ ]:
MERGE_SQL = (f"MERGE INTO {TABLE} t "
             f"USING (SELECT order_id, 'PAID' AS status, '{{writer}}' AS updated_by FROM {TABLE} WHERE order_id < 20) s "
             f"ON t.order_id = s.order_id "
             f"WHEN MATCHED THEN UPDATE SET status = s.status, updated_by = s.updated_by")


def merge_writer(writer_id, use_retry):
    ok, failed, retries = 0, 0, 0
    for _ in range(3):
        statement = MERGE_SQL.format(writer=f"writer_{writer_id}")
        try:
            if use_retry:
                retries += run_with_retry(statement)
            else:
                spark.sql(statement)
            ok += 1
        except Exception:
            failed += 1
    return writer_id, ok, failed, retries


with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(lambda w: merge_writer(w, use_retry=False), [1, 2, 3]))
for writer_id, ok, failed, _ in results:
    print(f"writer_{writer_id}: committed={ok} failed={failed}")

## 5. The same MERGE with the retry helper

With retries, every writer's MERGE eventually commits.

In [ ]:
with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(lambda w: merge_writer(w, use_retry=True), [1, 2, 3]))
for writer_id, ok, failed, retries in results:
    print(f"writer_{writer_id}: committed={ok} failed={failed} retries={retries}")

## 6. Check the table

Even when commits were rejected, the table never holds partial changes: there are still 20 seed rows plus the appended rows, and each committed MERGE produced one snapshot.

In [ ]:
spark.sql(f"SELECT updated_by, status, count(*) AS row_count FROM {TABLE} GROUP BY updated_by, status ORDER BY updated_by").show()
spark.sql(f"SELECT snapshot_id, operation, committed_at FROM {TABLE}.snapshots ORDER BY committed_at").show(50, truncate=False)

## Guidance

- **Concurrent appends:** safe without extra code.
- **Concurrent MERGE / UPDATE / DELETE that can touch the same rows:** wrap each statement in a retry loop with jittered backoff. Allow more attempts as you add writers; 10 is a reasonable start for 3 writers on separate clusters.
- **Reduce overlap where you can:** partition the work so writers update different partitions or key ranges, or schedule overlapping jobs one after another.
- Native Iceberg on AIDP uses Hadoop-style catalog metadata. For many concurrent writers or complex multi-engine writes, discuss your design with the AIDP team first.

## 7. Clean up

Run this cell to drop the demo schema and table.

In [ ]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG_NAME}.{SCHEMA_NAME} CASCADE")
print("Dropped", f"{CATALOG_NAME}.{SCHEMA_NAME}")